In [ ]:
# !pip install -q transformers datasets scikit-learn torch accelerate

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import torch
import torch.nn.functional as F
from torch import nn
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

# ============== CONFIG ==================
DATA_PATH = "/kaggle/input/focal-dataet/blp25_hatespeech_subtask_1A_train.tsv"  # your upload
DATA_PATH_2= "/kaggle/input/focal-dataet/blp25_hatespeech_subtask_1A_dev.tsv"  # your uploaded file
TEXT_COL = "text"     # <-- change if needed (e.g., "tweet", "sentence", etc.)
LABEL_COL = "label"   # you specified this
MODEL_NAME = "csebuetnlp/banglabert"  # or "roberta-base"
MAX_LEN = 128
TEST_SIZE = 0.15
RANDOM_STATE = 42
BATCH_SIZE = 16
NUM_EPOCHS = 10
LR = 2e-5
WEIGHT_DECAY = 0.01

# Focal loss hyperparams
GAMMA = 2.0                 # focusing parameter (common: 1.0–2.0)
ALPHA_MODE = "inv_freq"     # "inv_freq" | "uniform" | "custom"
CUSTOM_ALPHA = None         # e.g., [1.0, 2.0, 4.0] with length = num_labels
LABEL_SMOOTHING = 0.0       # set 0.0–0.1 for mild smoothing (optional)
# ========================================

# -------- Load data --------
train_df = pd.read_csv(DATA_PATH, sep="\t", keep_default_na=False)
val_df = pd.read_csv(DATA_PATH_2, sep="\t", keep_default_na=False)

# Map string labels to ids if needed
if train_df[LABEL_COL].dtype == object:
    label2id = {lbl: i for i, lbl in enumerate(sorted(train_df[LABEL_COL].unique()))}
    id2label = {i: lbl for lbl, i in label2id.items()}
    train_df[LABEL_COL] = train_df[LABEL_COL].map(label2id)
    val_df[LABEL_COL] = val_df[LABEL_COL].map(label2id)
else:
    classes_sorted = sorted(train_df[LABEL_COL].unique().tolist())
    label2id = {int(k): int(k) for k in classes_sorted}
    id2label = {int(k): str(int(k)) for k in classes_sorted}

num_labels = len(label2id)

# Stratified train/val split
# train_df, val_df = train_test_split(
#     df, test_size=TEST_SIZE, stratify=df[LABEL_COL], random_state=RANDOM_STATE
# )

# --------- Alpha (class weights) for focal loss (computed on TRAIN only) ---------
counts = train_df[LABEL_COL].value_counts().sort_index().reindex(range(num_labels), fill_value=0).values.astype(float)

if ALPHA_MODE == "inv_freq":
    # inverse frequency; normalize to keep average ≈ 1 (stable loss scale)
    inv = 1.0 / np.maximum(counts, 1.0)
    alpha = inv * (num_labels * (1.0 / inv.sum()))
elif ALPHA_MODE == "uniform":
    alpha = np.ones(num_labels, dtype=float)
elif ALPHA_MODE == "custom":
    if CUSTOM_ALPHA is None or len(CUSTOM_ALPHA) != num_labels:
        raise ValueError("Set CUSTOM_ALPHA to a list of length num_labels.")
    alpha = np.array(CUSTOM_ALPHA, dtype=float)
else:
    raise ValueError("ALPHA_MODE must be one of: inv_freq | uniform | custom")

alpha_tensor = torch.tensor(alpha, dtype=torch.float)

print("Class counts (train):", counts.tolist())
print("Alpha (per class):   ", alpha.tolist(), " | gamma:", GAMMA)

# --------- HF Datasets + tokenization ----------
train_ds = Dataset.from_pandas(train_df[[TEXT_COL, LABEL_COL]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[[TEXT_COL, LABEL_COL]], preserve_index=False)
raw = DatasetDict({"train": train_ds, "validation": val_ds})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

tok = raw.map(tokenize, batched=True)
tok = tok.rename_column(LABEL_COL, "labels")
tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# -------------- Model -------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id={v: k for k, v in id2label.items()},
)

# --------- Multi-class (softmax) Focal Loss -----------
class FocalLoss(nn.Module):
    """
    Multi-class focal loss with softmax.
    - logits: (N, C)
    - targets: (N,) int64 with values in [0..C-1]
    alpha: (C,) tensor of per-class weights (α)
    gamma: focusing parameter
    label_smoothing: optional float in [0, 1)
    """
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.label_smoothing = label_smoothing
        if alpha is not None:
            self.register_buffer("alpha", alpha.float())
        else:
            self.alpha = None

    def forward(self, logits, targets):
        # log softmax for numerical stability
        log_probs = F.log_softmax(logits, dim=-1)           # (N, C)
        probs = log_probs.exp()

        # one-hot targets (with optional smoothing)
        num_classes = logits.size(-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0)
            if self.label_smoothing > 0.0:
                eps = self.label_smoothing
                true_dist = true_dist * (1 - eps) + eps / num_classes

        # p_t and log(p_t)
        pt = (probs * true_dist).sum(dim=-1)               # (N,)
        log_pt = (log_probs * true_dist).sum(dim=-1)       # (N,)

        # focal modulating factor
        focal_factor = (1.0 - pt).clamp(min=1e-8).pow(self.gamma)

        # alpha weighting per example: α for the true class of each sample
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)        # (N,)
        else:
            alpha_t = 1.0

        loss = -alpha_t * focal_factor * log_pt            # (N,)
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss

# ---------- Custom Trainer to use focal loss -----------
class FocalLossTrainer(Trainer):
    def __init__(self, focal_loss_fn, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss_fn = focal_loss_fn

    # def compute_loss(self, model, inputs, return_outputs=False):
    #     labels = inputs.get("labels")
    #     outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
    #     logits = outputs.get("logits")
    #     loss = self.focal_loss_fn(logits, labels)
    #     return (loss, outputs) if return_outputs else loss

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        # separate out labels without mutating the caller's dict
        labels = inputs["labels"]
        model_inputs = {k: v for k, v in inputs.items() if k != "labels"}

        outputs = model(**model_inputs)
        logits = outputs.get("logits")

        # ensure focal loss is on the same device as logits
        self.focal_loss_fn = self.focal_loss_fn.to(logits.device)
        loss = self.focal_loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

# Instantiate focal loss (alpha on device is handled automatically by Trainer)
focal_loss = FocalLoss(
    alpha=alpha_tensor,
    gamma=GAMMA,
    label_smoothing=LABEL_SMOOTHING,
    reduction="mean"
)

# -------------- Metrics -----------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average="macro")
    acc = (preds == labels).mean()
    return {"macro_f1": macro_f1, "accuracy": acc}

# -------------- Training ----------------
training_args = TrainingArguments(
    output_dir="./outputs_focal",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

trainer = FocalLossTrainer(
    focal_loss_fn=focal_loss,
    model=model,
    args=training_args,
    train_dataset=tok["train"],
    eval_dataset=tok["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

# --------- Detailed evaluation ----------
preds = trainer.predict(tok["validation"])
y_true = preds.label_ids
y_pred = preds.predictions.argmax(axis=1)
target_names = [id2label[i] for i in range(num_labels)]
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))


2025-10-02 09:46:21.883433: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759398382.053634      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759398382.104256      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Class counts (train): [8212.0, 19954.0, 4227.0, 2331.0, 676.0, 122.0]
Alpha (per class):    [0.06949539850399944, 0.02860059198731299, 0.13501211557010726, 0.24482891999778783, 0.844225166442076, 4.677837807498717]  | gamma: 2.0


tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/2512 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_36/970043708.py:167: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalLossTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.044000,0.056031,0.473213,0.575239
2,0.029900,0.078359,0.511769,0.630971
3,0.018700,0.114578,0.518821,0.671178
4,0.013800,0.160474,0.522530,0.667994
5,0.017000,0.178356,0.518214,0.672771
